# 03. Класичні методи: кожен проти кожного пошкодження

**Тиждень 2.** Перевіряємо кожен метод з `src/enhance.py` окремо, а також класичний
пайплайн (завжди ті самі кроки). `otsu` тут антиприклад: Chen et al. (2019) показали,
що глобальний поріг при нерівному світлі гірший, ніж нічого не робити.

Прогін займає кілька хвилин.

In [ ]:
import sys
from pathlib import Path

# ноутбук лежить у notebooks/, корінь проєкту на рівень вище
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)

In [ ]:
from src.data import generate_synthetic
from src.enhance import ENHANCERS
from src.evaluate import run_benchmark, summarize

samples = generate_synthetic(n_per_format=8, seed=42)
methods = ["raw"] + list(ENHANCERS) + ["classic"]
print(methods)

In [ ]:
table = run_benchmark(samples, methods=methods)
table.to_csv(RESULTS / "03_enhancers.csv", index=False)
print("Рядків:", len(table))

## Приріст над raw: метод × пошкодження

Середній decode rate по рівнях 1-5 мінус те саме для `raw`. Зелене допомагає, червоне шкодить.

In [ ]:
DECODER = "zxing"   # TODO: подивись також "opencv" і "wechat"

damaged = table[(table["level"] > 0) & (table["decoder"] == DECODER)]
rates = summarize(damaged, by=["method", "degradation"])
grid = rates.pivot(index="method", columns="degradation", values="decode_rate")
gain = grid - grid.loc["raw"]

fig, ax = plt.subplots(figsize=(10, 6))
picture = ax.imshow(gain.values, cmap="RdYlGn", vmin=-0.6, vmax=0.6)
ax.set_xticks(range(len(gain.columns)))
ax.set_xticklabels(gain.columns, rotation=45, ha="right")
ax.set_yticks(range(len(gain.index)))
ax.set_yticklabels(gain.index)
for row in range(len(gain.index)):
    for col in range(len(gain.columns)):
        ax.text(col, row, str(round(gain.values[row, col], 2)), ha="center", va="center", fontsize=7)
ax.set_title("Приріст decode rate над raw, декодер " + DECODER)
plt.colorbar(picture)
plt.tight_layout()
plt.show()

## Як виглядає кадр після кожного методу (нерівне світло, рівень 4)

In [ ]:
from src.degradations import DEGRADATIONS
from src.evaluate import get_method

image = DEGRADATIONS["uneven"](samples[0]["image"], 4, seed=1)
fig, axes = plt.subplots(3, 4, figsize=(14, 7))
for ax, name in zip(axes.flat, methods):
    method = get_method(name)
    if method is None:
        shown = image
    else:
        shown = method(image)
    ax.imshow(shown, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Середній час на кадр (мс)

На лінії є ліміт часу на кадр, тож повільний метод може не підійти, навіть якщо він точніший.

In [ ]:
summarize(table, by=["method"]).sort_values("decode_rate", ascending=False)

**TODO для тебе:**
- Які методи допомагають проти яких пошкоджень? Чи збігається з теорією (лекції 2-4)?
- Чи підтвердився результат Chen et al. про `otsu`?
- Чому бінаризація (`otsu`, `sauvola`) може шкодити OpenCV? (згадай, що OpenCV не бачить ідеально різкі коди)